## Finetune Tokenizer

- https://huggingface.co/docs/tokenizers/python/latest/pipeline.html
- https://github.com/huggingface/notebooks/blob/main/examples/tokenizer_training.ipynb
- https://medium.com/@edandwe/my-journey-to-extend-a-3-billion-parameter-language-model-to-support-swedish-with-a-single-gpu-70dd3ab92748
- https://towardsdatascience.com/build-a-tokenizer-for-the-thai-language-from-scratch-0e4ea5f2a8b3/
- https://github.com/huggingface/tokenizers/blob/main/bindings/python/examples/custom_components.py

In [4]:
from tokenizers import Tokenizer, AddedToken, pre_tokenizers
from tokenizers.models import BPE, WordPiece, Unigram
from tokenizers.trainers import BpeTrainer, WordPieceTrainer, UnigramTrainer
from tokenizers.pre_tokenizers import PreTokenizer, Whitespace
from tokenizers.normalizers import NFD, Lowercase, StripAccents, Sequence

In [9]:

# --- 1. Initialisera en tokenizer med en modell (t.ex. BPE) ---
# Vi skapar en tom BPE-modell först
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

# --- 2. Ställ in normalisering ---
# Viktigt för svenska att hantera åäö korrekt.
# NFD Unicode-normalisering + Lowercase är vanligt.
# StripAccents kan ta bort prickar/ringar om du vill, men oftast vill du behålla åäö.
# Vi behåller åäö här genom att inte använda StripAccents.
tokenizer.normalizer = Sequence([NFD(), Lowercase()])
# Om du INTE vill ha gemener: tokenizer.normalizer = NFD()

# --- 3. Ställ in Pre-tokenizer ---
# Delar upp texten i "för-tokens" (oftast ord) som BPE sedan arbetar på.
# Whitespace är en vanlig startpunkt. ByteLevel kan vara bra om du vill ha ren byte-baserad BPE.
tokenizer.pre_tokenizer = Whitespace()

# --- 4. Ställ in Tränaren ---
# Här definierar du Vokabulärstorlek och specialtoken.
# vocab_size: Hur många subword-enheter vill du ha totalt?
# min_frequency: Hur många gånger måste ett par förekomma för att slås ihop?
# special_tokens: Mycket viktiga! Inkludera de token som din framtida Transformer-modell behöver.
vocab_size = 10000 # Typisk storlek, justera efter behov
trainer = BpeTrainer(
    vocab_size=vocab_size,
    min_frequency=2, # Ignorera väldigt ovanliga par
    special_tokens=[
        "[PAD]", # Padding-token
        "[UNK]", # Okänd token
        "[CLS]", # Klassificeringstoken (för BERT-liknande modeller)
        "[SEP]", # Separator-token (för BERT-liknande modeller)
        "[MASK]", # Maskeringstoken (för BERT-liknande modeller)
        # Lägg till fler om din modell kräver det
    ]
)


In [16]:
text = "Han behöver elmateral för sitt elarbete som elmontör."
tokenizer.pre_tokenizer.pre_tokenize_str(text)

[('Han', (0, 3)),
 ('behöver', (4, 11)),
 ('elmateral', (12, 21)),
 ('för', (22, 25)),
 ('sitt', (26, 30)),
 ('elarbete', (31, 39)),
 ('som', (40, 43)),
 ('elmontör', (44, 52)),
 ('.', (52, 53))]

In [10]:
# --- 5. Förbered träningsfiler ---
# Se till att filerna är UTF-8 kodade!
files = ["svensk_text_1.txt",
        "svensk_text_2.txt",
        "svensk_text_3.txt",
        ]


In [12]:
# --- 6. Träna tokenizern ---
# Tränar modellen (lär sig merge-regler och bygger vokabulären) från dina filer.
tokenizer.train(files, trainer)

# --- 7. Spara tokenizern ---
# Sparar konfigurationen och den tränade vokabulären till en JSON-fil.
output_path = "custom-svenska-tokenizer.json"
tokenizer.save(output_path)

print(f"Tokenizer tränad och sparad till {output_path}")
print(f"Vocab size after training: {tokenizer.get_vocab_size()}")




Tokenizer tränad och sparad till custom-svenska-tokenizer.json
Vocab size after training: 3727


In [14]:
# --- 8. Ladda och använd den tränade tokenizern ---
loaded_tokenizer = Tokenizer.from_file(output_path)

text = "Det här är en exempelmening på svenska med åäö och sammansatta ord som e-post."
text = "Elmontörer kommer att spela en viktig roll i framtidens samhälle."
text = "Elmaterial har en viktig roll i framtidens samhälle att spela."
# text = "Byggmontör kommer att spela en viktig roll i framtidens samhälle."
# text = "Byggarbete kommer att spela en viktig roll i framtidens samhälle."
# text = "Byggmaterial kommer att spela en viktig roll i framtidens samhälle."
text = "Han behöver elmateral för sitt elarbete som elmontör."
encoded = loaded_tokenizer.encode(text)

print(f"\nOriginaltext: '{text}'")
print("Token-IDs:", encoded.ids)
print("Tokens:", encoded.tokens)
print("Offsets:", encoded.offsets) # Visar var varje token börjar och slutar i originaltexten

# Avkoda tillbaka till text
decoded_text = loaded_tokenizer.decode(encoded.ids)
print("Avkodad text:", decoded_text)


Originaltext: 'Han behöver elmateral för sitt elarbete som elmontör.'
Token-IDs: [406, 732, 67, 105, 2304, 79, 1008, 1295, 125, 137, 13]
Tokens: ['han', 'behöver', 'el', 'mat', 'eral', 'för', 'sitt', 'elarbete', 'som', 'elmontör', '.']
Offsets: [(0, 3), (4, 11), (12, 14), (14, 17), (17, 21), (22, 25), (26, 30), (31, 39), (40, 43), (44, 52), (52, 53)]
Avkodad text: han behöver el mat eral för sitt elarbete som elmontör .
